# YOLO-Modell-Finetuning

Das vortrainierte YOLO-Modell wird auf den gegebenen GTSRB-Datensatz angepasst (Finetuning).

Hierzu wird der Klassifikationskopf des Modells durch einen vollvernetzten Layer ersetzt oder ergänzt, dessen Anzahl an Ausgabeneuronen der Anzahl der Klassen im Datensatz entspricht. Die Gewichte des restlichen Netzwerks bleiben dabei unverändert (eingefroren), sodass ausschließlich der neue Layer trainiert wird, um die Trainingszeit gering zu halten.

Weitere Informationen über das Trainieren des YOLO-Modells sind auf dieser Seite zu finden:

- https://docs.ultralytics.com/tasks/classify/#train

## Installation und Importe

Es wird die `ultralytics`-Bibliothek installiert und notwendige Python-Module importiert.

In [16]:
!pip install ultralytics -q

In [17]:
from sklearn.utils import shuffle
from ultralytics import YOLO
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pickle
import os
import cv2
import yaml

## Konfiguration der Laufzeitumgebung

Das Skript erkennt automatisch, ob es in Google Colab oder lokal ausgeführt wird, und bindet bei Bedarf Google Drive ein.

Die Beständigkeit der Daten ist wichtig. Da Colab-Laufzeiten temporär sind, ermöglicht die Anbindung von Google Drive den Zugriff auf große Datensätze, ohne diese jedes Mal neu hochladen zu müssen.

In [18]:
# Umgebung erkennen: Colab oder lokal
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE_PATH = '/content/drive/MyDrive/'  # ggf. an Projektordner anpassen
    print("Running in Google Colab – Google Drive wurde eingebunden.")
except ModuleNotFoundError:
    IN_COLAB = False
    BASE_PATH = './'  # lokales Arbeitsverzeichnis, ggf. anpassen
    print("Running locally – Google Drive wird nicht eingebunden.")

Running locally – Google Drive wird nicht eingebunden.


## Laden des GTSRB-Datensatzes

Es werden die Pfade zu den Trainings- und Validierungsdaten definiert und die vorverarbeiteten Pickle-Dateien geladen.

Um ein Modell zu trainieren, werden zwei getrennte Datensätze benötigt: den **Trainingsdatensatz**, aus dem das Modell lernt, und den **Validierungsdatensatz**, um die Genauigkeit auf ungesehenen Daten objektiv zu beurteilen und eine Überanpassung (Overfitting) zu erkennen.

In [19]:
# Zentrale Pfad-Konfiguration – abhängig von der Umgebung
if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/KI-thkoeln/praktikum_2/assets/GTSRB'
else:
    # Lokaler Pfad – bei Bedarf an eigene Ordnerstruktur anpassen
    DATA_DIR = os.path.join('dataset')

In [20]:
valid_dir = os.path.join(DATA_DIR, 'valid.p')
train_dir = os.path.join(DATA_DIR, 'train.p')

with open(valid_dir, mode='rb') as valid_dataset:
    valid = pickle.load(valid_dataset)
with open(train_dir, mode='rb') as train_dataset:
    train = pickle.load(train_dataset)

X_train, y_train = train["features"], train["labels"]
X_valid, y_valid = valid["features"], valid["labels"]

### Datensatz für YOLO vorbereiten

In diesem Schritt wird der GTSRB-Datensatz in eine Struktur transformiert, die mit dem Ultralytics YOLO-Framework kompatibel ist. Dies ist aus folgenden Gründen wichtig:

1.  YOLO (Classification) erwartet Bilder in Unterordnern, wobei jeder Ordnername der Klassen-ID entspricht.
2.  Es wird ein Dictionary (`gtsrb_classes_text`) erstellt, um den numerischen IDs verständliche Namen zuzuweisen (z.B. `1` zu `Speed limit (30km/h)`), was die spätere Interpretation der Ergebnisse erleichtert.
3.  Da die Daten ursprünglich in einem Python-Pickle-Format vorliegen, werden die Pixel-Arrays extrahiert und als `.png`-Dateien auf der Festplatte gespeichert, damit der YOLO-Dataloader sie effizient einlesen kann.
4.  Die YAML-Datei definiert, wo sich die Trainings- und Validierungsbilder befinden und wie die Klassen benannt sind.

In [21]:
# Mapping der GTSRB Klassen-IDs zu Text-Labels
gtsrb_classes_text = {
    0: 'Speed limit (20km/h)',
    1: 'Speed limit (30km/h)',
    2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)',
    4: 'Speed limit (70km/h)',
    5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)',
    7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)',
    9: 'No passing',
    10: 'No passing veh over 3.5t',
    11: 'Right-of-way at intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Veh > 3.5t prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve left',
    20: 'Dangerous curve right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows on the right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End speed + passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End of no passing',
    42: 'End no passing veh > 3.5t'
}

# Anzahl der Klassen aus den Trainingslabels berechnen
num_classes_gtsrb = len(set(y_train))
print(f"Anzahl der Klassen im GTSRB-Datensatz: {num_classes_gtsrb}")

# Basisverzeichnis für den Datensatz definieren
base_dataset_dir = 'yolo_gtsrb_dataset'
os.makedirs(base_dataset_dir, exist_ok=True)

# Train- und Validierungsverzeichnisse sowie Klassenunterverzeichnisse erstellen
train_img_base_dir = os.path.join(base_dataset_dir, 'train')
val_img_base_dir = os.path.join(base_dataset_dir, 'val')

for i in range(num_classes_gtsrb):
    os.makedirs(os.path.join(train_img_base_dir, str(i)), exist_ok=True)
    os.makedirs(os.path.join(val_img_base_dir, str(i)), exist_ok=True)

print("Speichere Trainingsbilder...")
for i, (image, label) in enumerate(zip(X_train, y_train)):
    class_folder = os.path.join(train_img_base_dir, str(label))
    cv2.imwrite(os.path.join(class_folder, f'{i:05d}.png'), image)
print(f"{len(X_train)} Trainingsbilder gespeichert.")

print("Speichere Validierungsbilder...")
for i, (image, label) in enumerate(zip(X_valid, y_valid)):
    class_folder = os.path.join(val_img_base_dir, str(label))
    cv2.imwrite(os.path.join(class_folder, f'{i:05d}.png'), image)
print(f"{len(X_valid)} Validierungsbilder gespeichert.")

data_config = {
    'path': base_dataset_dir,
    'train': 'train',
    'val': 'val',
    'names': gtsrb_classes_text
}

data_yaml_path = os.path.join(base_dataset_dir, 'data.yaml')
with open(data_yaml_path, 'w') as file:
    yaml.dump(data_config, file, default_flow_style=False)

print(f"data.yaml erstellt unter {data_yaml_path}")

Anzahl der Klassen im GTSRB-Datensatz: 43
Speichere Trainingsbilder...
34799 Trainingsbilder gespeichert.
Speichere Validierungsbilder...
4410 Validierungsbilder gespeichert.
data.yaml erstellt unter yolo_gtsrb_dataset/data.yaml


## Finetuning des YOLO-Klassifizierungsmodells

Beim Finetuning wird das Wissen eines bereits vortrainierten Modells (`yolo26n-cls.pt`) genutzt.

Anstatt bei Null anzufangen, wird das Modell nur an die spezifischen Verkehrszeichen angepasst. Das spart massiv Zeit und Rechenleistung.

Durch den Parameter `freeze=True` (oder das Einfrieren der unteren Layer) wird verhindert, dass die bereits gelernten allgemeinen Merkmale überschrieben werden. Nur der neue Klassifikationskopf am Ende des Netzwerks wird trainiert.

Das angepasste Modell wird im Ordner `/content/runs/classify/YOLO_GTSRB_cls_finetuning/yolo26n-cls_finetuned/weights/best.pt` gespeichert und kann dort heruntergeladen werden.

In [22]:
model = YOLO("yolo26n-cls.pt")

In [23]:
print(model.model) # Details zum Model

ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
  

In [24]:
print("Starte Finetuning des YOLO-Modells...")

# Trainiere das Modell auf dem generierten Datensatzordner
results = model.train(
    data="yolo_gtsrb_dataset",  # Pfad zum Datensatz-Ordner
    epochs=10,                  # Für ein Praktikum reichen 10-15 Epochen oft aus
    imgsz=32,                   # GTSRB Bildgröße (32x32)
    batch=64,                   # Batch-Größe
    freeze=10,                  # HIER GEÄNDERT: Reduziert auf 10, damit der Kopf trainiert werden kann
    project="YOLO_GTSRB_cls_finetuning",
    name="yolo26n-cls_finetuned"
)

print("Finetuning abgeschlossen.")

Starte Finetuning des YOLO-Modells...
Ultralytics 8.4.69 🚀 Python-3.9.25 torch-2.8.0 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_gtsrb_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=32, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n-cls_finetuned-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

In [25]:
print(model.model) # Details zum Model

ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
  

## Evaluierung des feinabgestimmten Modells

Nach dem Training muss objektiv gemessen werden, wie gut das Modell auf Daten performt, die während des Lernprozesses nicht gesehen wurden (der Validierungsdatensatz).

In [26]:
# Explizites Setzen der Klassennamen im Modell-Objekt für die Visualisierung
model.model.names = gtsrb_classes_text

# Evaluierung auf den Validierungsdaten, die in der data.yaml definiert sind
metrics = model.val()

print(f"Top-1 Accuracy auf Validierungsdaten: {metrics.top1 * 100:.2f} %")
print(f"Top-5 Accuracy auf Validierungsdaten: {metrics.top5 * 100:.2f} %")

Ultralytics 8.4.69 🚀 Python-3.9.25 torch-2.8.0 CPU (Apple M2)
YOLO26n-cls summary (fused): 47 layers, 1,581,107 parameters, 0 gradients, 3.2 GFLOPs
train: /Users/tim/Documents/PycharmProjects/ki-m2/yolo_gtsrb_dataset/train... found 34799 images in 43 classes ✅ 
val: /Users/tim/Documents/PycharmProjects/ki-m2/yolo_gtsrb_dataset/val... found 4410 images in 43 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 36.0±14.4 MB/s, size: 2.4 KB)
val: Scanning /Users/tim/Documents/PycharmProjects/ki-m2/yolo_gtsrb_dataset/val... 4410 images, 0 corrupt: 100% ━━━━━━━━━━━━ 4410/4410 1.5Git/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 276/276 15.1it/s 18.2s0.1s
                   all      0.465      0.832
Speed: 0.0ms preprocess, 3.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /opt/homebrew/runs/classify/val
Top-1 Accuracy auf Validierungsdaten: 46.46 %
Top-5 Accuracy auf Validierungsdaten: 83.15 %


### Visualisierung der Ergebnisse

Es werden die Trainingskurven (Loss/Accuracy) und die Confusion Matrix angezeigt, um den Trainingsverlauf zu bewerten.

Ein sinkender Loss zeigt, dass das Modell lernt, während die Confusion Matrix genau aufzeigt, bei welchen Klassen das Modell noch Schwierigkeiten hat (z.B. ähnliche Schilder).

In [27]:
train_results_dir = model.trainer.save_dir

results_png = os.path.join(model.trainer.save_dir, 'results.png')
if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Trainingsverlauf (Loss & Accuracy)')
    plt.show()

<Figure size 1200x800 with 1 Axes>

In [28]:
val_results_dir = metrics.save_dir

cm_png = os.path.join(metrics.save_dir, 'confusion_matrix.png')
if os.path.exists(cm_png):
    img_cm = mpimg.imread(cm_png)
    plt.figure(figsize=(12, 12))
    plt.imshow(img_cm)
    plt.axis('off')
    plt.title('YOLO Gefundene Konfusionsmatrix')
    plt.show()

<Figure size 1200x1200 with 1 Axes>

In [29]:

"""
TODO: Geben Sie die Top 10 Klassen mir Ihrer Häufigkeit aus.

Zum Beispiel:

Top 10 erkannte Klassen:
loupe: 1439
analog_clock: 223
...

"""

import cv2

top_classes = []

print("Starte Klassifizierung der Validierungsbilder...")
for img in X_valid:
    # YOLO benötigt oft BGR statt RGB
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # Vorhersage (Stream=True spart Arbeitsspeicher)
    results = model(img_bgr, verbose=False)

    for result in results:
        # Die Klasse mit der höchsten Wahrscheinlichkeit (Top-1) extrahieren
        top1_idx = result.probs.top1
        class_name = result.names[top1_idx]
        top_classes.append(class_name)

print(f"Erfolgreich {len(top_classes)} Bilder klassifiziert.")

# Häufigkeiten zählen
counter = Counter(top_classes)
top10 = counter.most_common(10)

print("\nTop 10 erkannte Klassen:")
print("-" * 30)
for klasse, anzahl in top10:
    print(f"{klasse}: {anzahl}")

Starte Klassifizierung der Validierungsbilder...
Erfolgreich 4410 Bilder klassifiziert.

Top 10 erkannte Klassen:
------------------------------
Priority road: 432
Speed limit (80km/h): 331
Speed limit (70km/h): 323
Turn left ahead: 311
Speed limit (30km/h): 306
Speed limit (60km/h): 221
Slippery road: 219
Keep right: 214
End speed + passing limits: 205
General caution: 194
